# Day 05: First Baseline Machine Learning Model

## Objective
Train and evaluate baseline classification models using the Breast Cancer Wisconsin Dataset.

## Models
1. DummyClassifier
2. Logistic Regression

## Evaluation Metrics
- Accuracy
- Precision
- Recall
- F1-Score

Random State: 42

 Import libraries

In [19]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

import os

RANDOM_STATE = 42

print("Libraries imported successfully!")

Libraries imported successfully!


Load dataset

In [20]:
df = pd.read_csv("../data/breast_cancer.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset Shape: (569, 31)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,malignant
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,malignant
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,malignant
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,malignant
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,malignant


Dataset inspection

In [21]:
print("Dataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nDuplicate Records:")
print(df.duplicated().sum())

print("\nTarget Distribution:")
print(df["diagnosis"].value_counts())

print("\nTarget Percentages:")
print(df["diagnosis"].value_counts(normalize=True) * 100)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoot

Separate features and target

In [22]:
X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nFeature Columns:")
print(X.columns.tolist())

print("\nTarget Classes:")
print(y.unique())

Features Shape: (569, 30)
Target Shape: (569,)

Feature Columns:
['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension']

Target Classes:
['malignant' 'benign']


Train, validation and test split

In [23]:
# First split: 70% training and 30% temporary data

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

# Second split: divide temporary data equally
# 15% validation and 15% testing

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print("Training Set:", X_train.shape)
print("Validation Set:", X_val.shape)
print("Testing Set:", X_test.shape)

print("\nSplitting completed successfully!")

Training Set: (398, 30)
Validation Set: (85, 30)
Testing Set: (86, 30)

Splitting completed successfully!


Verify class distribution

In [24]:
print("Training Class Distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nValidation Class Distribution:")
print(y_val.value_counts(normalize=True) * 100)

print("\nTesting Class Distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training Class Distribution:
diagnosis
benign       62.81407
malignant    37.18593
Name: proportion, dtype: float64

Validation Class Distribution:
diagnosis
benign       62.352941
malignant    37.647059
Name: proportion, dtype: float64

Testing Class Distribution:
diagnosis
benign       62.790698
malignant    37.209302
Name: proportion, dtype: float64


Create baseline models

In [25]:
# Model 1: Dummy Classifier

dummy_model = DummyClassifier(
    strategy="most_frequent"
)

# Model 2: Logistic Regression Pipeline

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

print("Both baseline models created successfully!")

Both baseline models created successfully!


Train the models

In [26]:
# Train Dummy Classifier

dummy_model.fit(X_train, y_train)

# Train Logistic Regression

logistic_model.fit(X_train, y_train)

print("Dummy Classifier trained successfully!")
print("Logistic Regression trained successfully!")

Dummy Classifier trained successfully!
Logistic Regression trained successfully!


Create evaluation function

In [27]:
def evaluate_model(model, X_data, y_data, model_name):

    predictions = model.predict(X_data)

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_data, predictions),
        "Precision": precision_score(
            y_data, predictions,
            pos_label="malignant",
            zero_division=0
        ),
        "Recall": recall_score(
            y_data, predictions,
            pos_label="malignant",
            zero_division=0
        ),
        "F1 Score": f1_score(
            y_data, predictions,
            pos_label="malignant",
            zero_division=0
        )
    }

    return metrics

Compare baseline models on validation data

In [28]:
dummy_results = evaluate_model(
    dummy_model, X_val, y_val, "Dummy Classifier"
)

logistic_results = evaluate_model(
    logistic_model, X_val, y_val, "Logistic Regression"
)

validation_results = pd.DataFrame([
    dummy_results,
    logistic_results
])

print("BASELINE MODEL COMPARISON")
display(validation_results.round(4))

BASELINE MODEL COMPARISON


,Model,Accuracy,Precision,Recall,F1 Score
0,Dummy Classifier,0.6235,0.0,0.0000,0.0000
1,Logistic Regression,0.9647,1.0,0.9062,0.9508


Final test evaluation

In [29]:
final_test_results = evaluate_model(
    logistic_model,
    X_test,
    y_test,
    "Logistic Regression - Final Test"
)

print("FINAL TEST RESULTS")
display(pd.DataFrame([final_test_results]).round(4))

FINAL TEST RESULTS


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression - Final Test,0.9767,0.9688,0.9688,0.9688


Classification report

In [30]:
y_test_predictions = logistic_model.predict(X_test)

print("Classification Report:")
print(
    classification_report(
        y_test,
        y_test_predictions,
        labels=["benign", "malignant"],
        zero_division=0
    )
)

Classification Report:
              precision    recall  f1-score   support

      benign       0.98      0.98      0.98        54
   malignant       0.97      0.97      0.97        32

    accuracy                           0.98        86
   macro avg       0.98      0.98      0.98        86
weighted avg       0.98      0.98      0.98        86



In [31]:
# Combine validation comparison and final test results

final_results = pd.concat(
    [
        validation_results,
        pd.DataFrame([final_test_results])
    ],
    ignore_index=True
)

# Create reports directory

reports_dir = PROJECT_DIR / "reports"
reports_dir.mkdir(exist_ok=True)

# Save metrics

metrics_path = reports_dir / "baseline_metrics.csv"

final_results.to_csv(metrics_path, index=False)

print("Metrics saved successfully!")
print("File:", metrics_path)

display(final_results.round(4))

Metrics saved successfully!
File: c:\Users\ASUS\OneDrive\Desktop\20-Day-AI-ML-Internship\Day-05\reports\baseline_metrics.csv


,Model,Accuracy,Precision,Recall,F1 Score
0,Dummy Classifier,0.6235,0.0000,0.0000,0.0000
1,Logistic Regression,0.9647,1.0000,0.9062,0.9508
2,Logistic Regression - Final Test,0.9767,0.9688,0.9688,0.9688


In [32]:
from pathlib import Path

PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "data").exists():
    PROJECT_DIR = PROJECT_DIR.parent

print("Project Directory:", PROJECT_DIR)
print("Reports Directory:", PROJECT_DIR / "reports")

Project Directory: c:\Users\ASUS\OneDrive\Desktop\20-Day-AI-ML-Internship\Day-05
Reports Directory: c:\Users\ASUS\OneDrive\Desktop\20-Day-AI-ML-Internship\Day-05\reports


In [33]:
print(list((PROJECT_DIR / "reports").iterdir()))

[WindowsPath('c:/Users/ASUS/OneDrive/Desktop/20-Day-AI-ML-Internship/Day-05/reports/baseline_metrics.csv'), WindowsPath('c:/Users/ASUS/OneDrive/Desktop/20-Day-AI-ML-Internship/Day-05/reports/baseline_model_report.md')]


In [34]:
display(validation_results.round(4))

,Model,Accuracy,Precision,Recall,F1 Score
0,Dummy Classifier,0.6235,0.0,0.0000,0.0000
1,Logistic Regression,0.9647,1.0,0.9062,0.9508


In [35]:
display(pd.DataFrame([final_test_results]).round(4))

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression - Final Test,0.9767,0.9688,0.9688,0.9688


## Conclusion

Successfully trained and evaluated DummyClassifier and Logistic Regression models using Scikit-learn.

Logistic Regression achieved 96.47% validation accuracy and 97.67% test accuracy.

This project helped understand baseline model training, evaluation metrics, pipelines, and reproducibility.